In [2]:
import os
os.makedirs("assignment2_23K0594", exist_ok=True)
os.chdir("assignment2_23K0594")
print("Working directory:", os.getcwd())

Working directory: /content/assignment2_23K0594


In [3]:
#installing library
!pip install ultralytics

In [4]:
#downloading
import os
os.makedirs("coco/annotations", exist_ok=True)

#download images
!wget -q http://images.cocodataset.org/zips/val2017.zip
!unzip -q val2017.zip -d coco/
print("Images downloaded ✅")

#download annotations
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q annotations_trainval2017.zip -d coco/
print("Annotations downloaded ✅")

Images downloaded ✅
Annotations downloaded ✅


In [9]:
import os, json, shutil
from pathlib import Path
from collections import defaultdict

os.chdir("/content/assignment2_23K0594")

SELECTED_CLASSES = ["person", "car", "dog", "cat", "bicycle", "chair", "bottle"]
COCO_IMAGES_DIR  = "coco/val2017"
COCO_ANN_FILE    = "coco/annotations/instances_val2017.json"
OUTPUT_DIR       = "coco_subset"

print("Loading annotations...")
with open(COCO_ANN_FILE) as f:
    coco = json.load(f)

name_to_catid    = {c["name"]: c["id"] for c in coco["categories"]}
selected_catids  = [name_to_catid[n] for n in SELECTED_CLASSES]
catid_to_yoloidx = {cid: i for i, cid in enumerate(selected_catids)}

relevant_image_ids = set()
for ann in coco["annotations"]:
    if ann["category_id"] in selected_catids:
        relevant_image_ids.add(ann["image_id"])

print(f"Found {len(relevant_image_ids)} relevant images")

for split in ["images/train","images/val","labels/train","labels/val"]:
    os.makedirs(f"{OUTPUT_DIR}/{split}", exist_ok=True)

image_ids_list = sorted(list(relevant_image_ids))
split_idx      = int(len(image_ids_list) * 0.8)
train_ids      = set(image_ids_list[:split_idx])
val_ids        = set(image_ids_list[split_idx:])
image_id_info  = {img["id"]: img for img in coco["images"]}

anns_by_image = defaultdict(list)
for ann in coco["annotations"]:
    if ann["category_id"] in selected_catids and ann["image_id"] in relevant_image_ids:
        anns_by_image[ann["image_id"]].append(ann)

def coco_to_yolo(bbox, w, h):
    x,y,bw,bh = bbox
    return (x+bw/2)/w, (y+bh/2)/h, bw/w, bh/h

print("Copying images and writing labels...")
for img_id in image_ids_list:
    info  = image_id_info[img_id]
    fname = info["file_name"]
    split = "train" if img_id in train_ids else "val"
    src   = f"{COCO_IMAGES_DIR}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, f"{OUTPUT_DIR}/images/{split}/{fname}")
    with open(f"{OUTPUT_DIR}/labels/{split}/{Path(fname).stem}.txt","w") as lf:
        for ann in anns_by_image[img_id]:
            cx,cy,nw,nh = coco_to_yolo(ann["bbox"], info["width"], info["height"])
            lf.write(f"{catid_to_yoloidx[ann['category_id']]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")

yaml = f"""path: /content/assignment2_23K0594/coco_subset
train: images/train
val:   images/val
nc: 7
names: {SELECTED_CLASSES}
"""
with open("coco_subset.yaml","w") as f:
    f.write(yaml)

print("Dataset ready ✅")
print(f"Train: {len(train_ids)} | Val: {len(val_ids)}")
print("\nYAML contents:")
print(yaml)

Loading annotations...
Found 3421 relevant images
Copying images and writing labels...
Dataset ready ✅
Train: 2736 | Val: 685

YAML contents:
path: /content/assignment2_23K0594/coco_subset
train: images/train
val:   images/val
nc: 7
names: ['person', 'car', 'dog', 'cat', 'bicycle', 'chair', 'bottle']



In [10]:
from ultralytics import YOLO
import os

os.chdir("/content/assignment2_23K0594")

model   = YOLO("yolov8n.pt")
results = model.train(
    data    = "/content/assignment2_23K0594/coco_subset.yaml",
    epochs  = 30,
    imgsz   = 640,
    batch   = 16,
    device  = "cuda",
    project = "/content/assignment2_23K0594/runs/detect",
    name    = "train_23K0594",
    plots   = True,
    workers = 2,
)
print("Training completed!!")
print("Best weights: /content/assignment2_23K0594/runs/detect/train_23K0594/weights/best.pt")

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/assignment2_23K0594/coco_subset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_23K0594, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

In [11]:
model   = YOLO("runs/detect/train_23K0594/weights/best.pt")
metrics = model.val(data="coco_subset.yaml", imgsz=640, conf=0.25, iou=0.5)

CLASSES = ["person","car","dog","cat","bicycle","chair","bottle"]
print(f"\n{'Class':<12}{'Precision':>12}{'Recall':>10}{'mAP@0.5':>10}")
print("-" * 45)
for i, cls in enumerate(CLASSES):
    print(f"{cls:<12}{metrics.box.p[i]:>12.4f}{metrics.box.r[i]:>10.4f}{metrics.box.ap50[i]:>10.4f}")
print("-" * 45)
print(f"{'Overall':<12}{metrics.box.mp:>12.4f}{metrics.box.mr:>10.4f}{metrics.box.map50:>10.4f}")

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2041.4±583.8 MB/s, size: 134.9 KB)
val: Scanning /content/assignment2_23K0594/coco_subset/labels/val.cache... 685 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 685/685 205.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 43/43 5.6it/s 7.7s
                   all        685       3162      0.596      0.416      0.524      0.367
                person        533       2037      0.758      0.645      0.746      0.529
                   car        109        385       0.62      0.343      0.496      0.326
                   dog         36         42      0.476      0.476       0.54      0.428
                   cat         33         37      0.774      0.649      0.723      0.529
               bicycle   

In [12]:
import glob, random
from PIL import Image as PILImage
import numpy as np

model      = YOLO("runs/detect/train_23K0594/weights/best.pt")
val_imgs   = glob.glob("coco_subset/images/val/*.jpg")
selected   = random.sample(val_imgs, 10)
os.makedirs("detection_outputs", exist_ok=True)

for i, img_path in enumerate(selected, 1):
    result     = model(img_path, conf=0.25, verbose=False)[0]
    annotated  = result.plot()
    pil_img    = PILImage.fromarray(annotated[:,:,::-1])
    pil_img.save(f"detection_outputs/detection_{i:02d}.jpg")
    print(f"Saved detection_{i:02d}.jpg")

print("Visualization completed!")

Saved detection_01.jpg
Saved detection_02.jpg
Saved detection_03.jpg
Saved detection_04.jpg
Saved detection_05.jpg
Saved detection_06.jpg
Saved detection_07.jpg
Saved detection_08.jpg
Saved detection_09.jpg
Saved detection_10.jpg
Visualization complete ✅


In [13]:
import glob, random, json
from PIL import Image, ImageDraw

CLASSES     = ["person","car","dog","cat","bicycle","chair","bottle"]
VAL_IMAGES  = "coco_subset/images/val"
VAL_LABELS  = "coco_subset/labels/val"
os.makedirs("error_outputs", exist_ok=True)

def iou(a, b):
    xA,yA = max(a[0],b[0]), max(a[1],b[1])
    xB,yB = min(a[2],b[2]), min(a[3],b[3])
    inter  = max(0,xB-xA)*max(0,yB-yA)
    return inter/((a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter+1e-6)

def yolo_box(cx,cy,w,h,W,H):
    return [(cx-w/2)*W,(cy-h/2)*H,(cx+w/2)*W,(cy+h/2)*H]

model    = YOLO("runs/detect/train_23K0594/weights/best.pt")
imgs     = glob.glob(f"{VAL_IMAGES}/*.jpg")
sample   = random.sample(imgs, min(200, len(imgs)))
tp=fp=fn = 0
examples = []

for img_path in sample:
    img       = Image.open(img_path)
    W, H      = img.size
    stem      = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path  = f"{VAL_LABELS}/{stem}.txt"
    gts       = []
    if os.path.exists(lbl_path):
        for line in open(lbl_path):
            p = line.split()
            gts.append({"cls":int(p[0]),"box":yolo_box(*map(float,p[1:]),W,H),"matched":False})

    preds = []
    res   = model(img_path, conf=0.25, verbose=False)[0]
    if res.boxes:
        for b in res.boxes:
            preds.append({"cls":int(b.cls[0]),"conf":float(b.conf[0]),"box":b.xyxy[0].tolist(),"matched":False})

    has_err = False
    for pred in preds:
        best, best_gt = 0, None
        for gt in gts:
            v = iou(pred["box"], gt["box"])
            if v > best: best, best_gt = v, gt
        if best >= 0.5 and best_gt and not best_gt["matched"] and pred["cls"]==best_gt["cls"]:
            best_gt["matched"] = pred["matched"] = True
            tp += 1
        else:
            fp += 1; has_err = True
    for gt in gts:
        if not gt["matched"]:
            fn += 1; has_err = True
    if has_err and len(examples) < 5:
        examples.append((img_path, preds, gts))

print(f"\nTrue Positives  : {tp}")
print(f"False Positives : {fp}")
print(f"False Negatives : {fn}")
print(f"Precision       : {tp/(tp+fp+1e-6):.4f}")
print(f"Recall          : {tp/(tp+fn+1e-6):.4f}")

for idx,(img_path,preds,gts) in enumerate(examples):
    img  = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    for gt in gts:
        col = (0,200,0) if not gt["matched"] else (180,180,180)
        draw.rectangle(gt["box"], outline=col, width=2)
        draw.text((gt["box"][0],gt["box"][1]), f"GT:{CLASSES[gt['cls']]}", fill=col)
    for p in preds:
        col = (0,120,255) if p["matched"] else (255,60,60)
        draw.rectangle(p["box"], outline=col, width=2)
        draw.text((p["box"][0],p["box"][1]+12), f"P:{CLASSES[p['cls']]}", fill=col)
    img.save(f"error_outputs/error_{idx+1}.jpg")

print("Error images saved ✅  (Green=missed GT | Red=FP | Blue=correct)")


True Positives  : 454
False Positives : 261
False Negatives : 523
Precision       : 0.6350
Recall          : 0.4647
Error images saved ✅  (Green=missed GT | Red=FP | Blue=correct)


In [21]:
from google.colab import files
import shutil, os

#downloading weights
files.download("/content/assignment2_23K0594/runs/detect/train_23K0594/weights/best.pt")

shutil.make_archive(
    "/content/assignment2_23K0594/outputs_only",
    "zip",
    "/content/assignment2_23K0594/detection_outputs"
)

files.download("/content/assignment2_23K0594/outputs_only.zip")
print("Done downloading!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done ✅
